In [1]:
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np

/home/scur0412/NeuroNLP/neuro/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⚠️ It's very important to run the cell below on Snellius. It'll ensure that model weights are downloaded in the scratch space instead of taking up all your user space.

In [20]:
! export HF_HOME='/scratch-shared/scur0412/hf_dir/'
! export HF_HUB_CACHE='/scratch-shared/scur0412/hf_dir/'

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("princeton-nlp/sup-simcse-bert-base-uncased")
model = AutoModel.from_pretrained("princeton-nlp/sup-simcse-bert-base-uncased").to(device)

In [22]:
my_sentence = "Roses are red, violets are blue\nI should get sentence representations\nbut I don't have a clue."
print(my_sentence)

inputs = tokenizer(my_sentence, padding=True, truncation=True, return_tensors="pt")

Roses are red, violets are blue
I should get sentence representations
but I don't have a clue.


In [23]:
with torch.no_grad():
    embeddings = model(**inputs.to(device), output_hidden_states=True, return_dict=True)

In [24]:
embeddings.keys()

odict_keys(['last_hidden_state', 'pooler_output', 'hidden_states'])

In [34]:
# the hidden states of the last model layer
embeddings['last_hidden_state'].shape

torch.Size([4, 11, 768])

⚠️ This `last_hidden_state` refers to the 'hidden states from _all_ tokens, extrated from the last model layer'. It is different from the hidden state of the last token of the sequence!

In [ ]:
len(embeddings['hidden_states']) #number of layers in the model

13

In [27]:
embeddings['hidden_states'][0].shape

torch.Size([1, 24, 768])

Normally, we'd be interested in the last hidden state of the last token of the sequence. Hoever, for this model only (since it's based on the BERT architecture), it makes more sense to consider the hidden states of the _first_ token of the sequence (the CLS token).

In [28]:
cls_embeddings = np.stack([hs[0,0,:].detach().cpu().numpy() for hs in embeddings['hidden_states']])

In [29]:
cls_embeddings.shape

(13, 768)

Now we have a 768-dimensional embedding for each of the 13 model layers.

Let's look at one more thing to pay attention to when extracting embeddings from multiple sentences.

In [30]:
sentences = ["Roses are red", "violets are blue", "I should get sentence representations", "but I don't have a clue."]
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")


In [31]:
with torch.no_grad():
    embeddings = model(**inputs.to(device), output_hidden_states=True, return_dict=True)

In [32]:
embeddings['hidden_states'][0].shape

torch.Size([4, 11, 768])

Now we have 11-token representations for all four sentences, which, however, don't have the same lenght. This is because some padding was applied.

If we want to consider the hidden state of the last token of the sequence as our sentence representation, it is _extremely_ important to select the right token, and not a padding one.

In [33]:
inputs['attention_mask']

tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

We can use the attention mask to filter out only relevant tokens—zeros are placed where padding tokens have been applied.

In [16]:
att_mask = inputs['attention_mask'].detach().cpu().numpy().astype(bool)
att_mask

array([[ True,  True,  True,  True,  True, False, False, False, False,
        False, False],
       [ True,  True,  True,  True,  True,  True, False, False, False,
        False, False],
       [ True,  True,  True,  True,  True,  True,  True, False, False,
        False, False],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True]])

In [17]:
att_mask.shape

(4, 11)

In [18]:
embeddings['hidden_states'][0].detach().cpu().numpy().shape

(4, 11, 768)

In [19]:
embeddings['hidden_states'][0].detach().cpu().numpy()[3][att_mask[3]].shape

(11, 768)